# Conclusion final v03 - NASA C-MAPSS RUL

Cierre documental post multi-prefix / EOL smoothing. No entrena modelos, no genera predicciones nuevas y no usa test final para decidir modelos.

## A. Seleccion final de modelos

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "results" / "resumen_global_final_modelos_v03.csv").exists():
        PROJECT_ROOT = candidate
        break

summary = pd.read_csv(PROJECT_ROOT / "results/resumen_global_final_modelos_v03.csv")
selection = pd.read_csv(PROJECT_ROOT / "results/seleccion_final_modelos_v03.csv")
registry = pd.read_csv(PROJECT_ROOT / "results/model_registry_final_v03.csv")
bins = pd.read_csv(PROJECT_ROOT / "results/metricas_por_bin_final_modelos_v03.csv")
diagnostics = pd.read_csv(PROJECT_ROOT / "results/diagnostics_final_modelos_v03.csv")
fd002_preds = pd.read_csv(PROJECT_ROOT / "results/FD002/fd002_multiprefix_predictions_v01.csv")
fd004_preds = pd.read_csv(PROJECT_ROOT / "results/FD004/FD004_multiprefix_late_stage_guard_predictions_v02.csv")
display(selection)
display(registry)

## B. Explicacion de los modelos

- **FD001**: LightGBM quantile alpha 0.40, RUL cap 125 y ventana temporal 50. Queda congelado.
- **FD002**: XGBoost condition/fault-sensitive como base; adopta multi-prefix `eol_mean`. No es offset: suaviza el ciclo estimado de falla.
- **FD003**: LightGBM w50 cap125 quantile alpha 0.40 `none_fault_sensitive`; offset 0 porque offsets positivos empeoraron CMAPSS/dangerous.
- **FD004**: XGBoost w70 bin_weights como base; adopta multi-prefix `eol_mean` con late-stage guard: si `baseline_pred <= 60` y `eol_mean_pred < baseline_pred`, usa `max(eol_mean_pred, baseline_pred - 2)`.

## C. Resultados cuantitativos

In [ ]:
cols = ["dataset", "selected_candidate", "MAE", "RMSE", "R2", "CMAPSS_mean", "dangerous_20", "bias", "adoption_reason"]
display(summary[cols])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
summary.plot.bar(x="dataset", y="RMSE", ax=axes[0], legend=False, title="RMSE final por FD")
summary.plot.bar(x="dataset", y="CMAPSS_mean", ax=axes[1], legend=False, title="CMAPSS mean final por FD")
summary.plot.bar(x="dataset", y="dangerous_20", ax=axes[2], legend=False, title="dangerous_20 final por FD")
for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

fd002_compare = pd.read_csv(PROJECT_ROOT / "results/FD002/fd002_multiprefix_eol_smoothing_results_v01.csv")
fd002_compare = fd002_compare[fd002_compare["candidate_name"].isin(["baseline", "eol_mean"])]
fd004_compare = pd.read_csv(PROJECT_ROOT / "results/FD004/FD004_multiprefix_late_stage_guard_results_v02.csv")
fd004_compare = fd004_compare[fd004_compare["candidate_name"].isin(["baseline", "eol_mean_guard_b60_only_if_lower_floor2"])]
display(Markdown("### FD002 baseline vs final"))
display(fd002_compare[["candidate_name", "RMSE", "CMAPSS_mean", "dangerous_20_pct", "MAE_RUL_le_50"]])
display(Markdown("### FD004 baseline vs final"))
display(fd004_compare[["candidate_name", "RMSE", "CMAPSS_mean", "dangerous_20_pct", "MAE_RUL_le_50"]])

def delta_table(df, final_name):
    base = df[df["candidate_name"].eq("baseline")].iloc[0]
    final = df[df["candidate_name"].eq(final_name)].iloc[0]
    return pd.DataFrame([{
        "final_candidate": final_name,
        "delta_RMSE_pct": (final["RMSE"] - base["RMSE"]) / base["RMSE"] * 100,
        "delta_CMAPSS_mean_pct": (final["CMAPSS_mean"] - base["CMAPSS_mean"]) / base["CMAPSS_mean"] * 100,
        "delta_dangerous_20_pct_points": final["dangerous_20_pct"] - base["dangerous_20_pct"],
    }])
display(pd.concat([
    delta_table(fd002_compare, "eol_mean").assign(dataset="FD002"),
    delta_table(fd004_compare, "eol_mean_guard_b60_only_if_lower_floor2").assign(dataset="FD004"),
], ignore_index=True))

## D. Resultados por rango de RUL

In [ ]:
display(bins[["dataset", "candidate_name", "rul_bin", "n_eval_points", "MAE", "CMAPSS_mean", "CMAPSS_share", "dangerous_20_pct", "source_bin_definition"]])

for dataset in ["FD002", "FD004"]:
    plot_bins = bins[(bins["dataset"].eq(dataset)) & (bins["candidate_name"].isin(["eol_mean", "eol_mean_guard_b60_only_if_lower_floor2"]))]
    if plot_bins.empty:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    plot_bins.plot.bar(x="rul_bin", y="MAE", ax=axes[0], legend=False, title=f"{dataset}: MAE por bin RUL")
    plot_bins.plot.bar(x="rul_bin", y="dangerous_20_pct", ax=axes[1], legend=False, title=f"{dataset}: dangerous_20 por bin RUL")
    plt.tight_layout()
    plt.show()

fd002_plot = fd002_preds[fd002_preds["candidate_name"].eq("eol_mean")]
fd004_plot = fd004_preds[fd004_preds["candidate_name"].eq("eol_mean_guard_b60_only_if_lower_floor2")]
for dataset, frame in [("FD002", fd002_plot), ("FD004", fd004_plot)]:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(frame["true_RUL"], frame["pred_RUL"], s=12, alpha=0.45)
    axes[0].plot([0, frame["true_RUL"].max()], [0, frame["true_RUL"].max()], color="black", linewidth=1)
    axes[0].set_title(f"{dataset}: true vs pred final")
    axes[0].set_xlabel("True RUL")
    axes[0].set_ylabel("Pred RUL")
    axes[1].scatter(frame["true_RUL"], frame["error"], s=12, alpha=0.45)
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_title(f"{dataset}: residual final")
    axes[1].set_xlabel("True RUL")
    axes[1].set_ylabel("Pred - true")
    plt.tight_layout()
    plt.show()

## E. Interpretacion metodologica

- **RUL cap 125**: prioriza la zona util cercana/media a falla y sacrifica precision en motores muy sanos.
- **Quantile alpha 0.40**: empuja modelos LightGBM a ser mas conservadores y reduce riesgo de sobreestimacion.
- **Condition-sensitive**: en FD002/FD004 controla regimen operativo mediante normalizacion/contexto por condicion y features temporales.
- **Fault-sensitive**: captura patrones de degradacion compatibles con modos de falla, sin afirmar etiquetas reales de fault mode.
- **Ventana temporal**: resume sensores con ultimos valores, medias, pendientes, deltas y variabilidad.
- **Multi-prefix**: para cada ciclo evaluado usa prefijos anteriores o iguales, convierte cada prediccion a ciclo estimado de falla y suaviza ese ciclo. No promedia RUL directamente.

## F. Riesgos y limitaciones

- No se uso test final para seleccionar ni calibrar.
- La validacion es interna por motores/cortes artificiales.
- Hay riesgo de sobreajuste si se agregan demasiadas reglas post-smoothing.
- El guard FD004 es una regla de inferencia/calibracion simple, no un modelo nuevo.
- La cola de RUL alto sigue siendo dificil por RUL cap y por menor historia previa util.
- C-MAPSS penaliza mas errores tardios/peligrosos, por eso dangerous_20 y CMAPSS pesan mas que mejoras minimas de RMSE.

## G. Conclusion final

FD001 y FD003 quedan con modelos LightGBM quantile conservadores. FD002 y FD004 quedan con modelos base XGBoost reforzados por inferencia temporal multi-prefix. La mejora clave del cierre fue pasar de offsets a suavizado del ciclo estimado de falla. FD004 ademas requiere un guard cercano a falla para no degradar la zona critica.

## Chequeos finales

In [ ]:
created_files = [
    "results/resumen_global_final_modelos_v03.csv",
    "results/seleccion_final_modelos_v03.csv",
    "results/model_registry_final_v03.csv",
    "results/metricas_por_bin_final_modelos_v03.csv",
    "results/diagnostics_final_modelos_v03.csv",
    "notebooks/conclusion/notebook_conclusion_final_v03.ipynb",
    "notas/notas_modelado_final_v03.txt",
    "notas/notas_conclusiones_final_v03.txt",
    "notas/notas_resultados_finales_v03.txt",
]
backup_files = ["notebooks/conclusion/notebook_conclusion_final_actualizado_backup_pre_v03.ipynb"]
display(pd.DataFrame({"created_files": created_files}))
display(pd.DataFrame({"backup_files": backup_files}))
display(selection[["dataset", "selected_candidate"]])
display(Markdown("No se uso test final para decisiones v03."))
display(registry[["dataset", "uses_final_test", "no_future_prefixes_used"]])
nan_summary = summary.isna().sum().reset_index()
nan_summary.columns = ["column", "n_nan"]
display(nan_summary[nan_summary["n_nan"] > 0])
display(diagnostics)